<a href="https://colab.research.google.com/github/Ansh1657/MRI-Deepfake-Detection-System/blob/main/notebooks/01_densenet121_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import glob
import os
import time

Dataset Definition

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, real_folders, fake_folders, transform=None):
        self.image_paths = []
        self.labels = []
        self.transform = transform

        # Label 0: Real Images
        for folder in real_folders:
            if os.path.exists(folder):
                paths = self._get_images(folder)
                self.image_paths.extend(paths)
                self.labels.extend([0.0] * len(paths))
            else:
                print(f"Warning: Real folder not found -> {folder}")

        # Label 1: Fake Images
        for folder in fake_folders:
            if os.path.exists(folder):
                paths = self._get_images(folder)
                self.image_paths.extend(paths)
                self.labels.extend([1.0] * len(paths))
            else:
                print(f"Warning: Fake folder not found -> {folder}")

        print(f"Total Dataset: {self.labels.count(0.0)} Real | {self.labels.count(1.0)} Fake")

    def _get_images(self, folder):
        paths = []
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            paths.extend(glob.glob(os.path.join(folder, ext)))
            paths.extend(glob.glob(os.path.join(folder, ext.upper())))
        return paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor([label], dtype=torch.float32)

Architecture Setup


In [ ]:
def build_densenet_detector():
    # Load pre-trained DenseNet-121 (trained on ImageNet)
    model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

    # Freeze early layers to prevent overfitting on your small dataset
    for param in model.features.parameters():
        param.requires_grad = False

    # Unfreeze the last denseblock for fine-tuning to MRI textures
    for param in model.features.denseblock4.parameters():
        param.requires_grad = True

    # Replace the final classification layer for Binary output (Real vs Fake)
    num_ftrs = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Linear(num_ftrs, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, 1)  # 1 output node for Binary Cross Entropy with Logits
    )

    return model

Training Loop

In [ ]:
def train_detector(real_dirs, fake_dirs, save_path, epochs=20, batch_size=32):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Targeting device: {device}")

    # Standard ImageNet normalization required for pre-trained torchvision models
    transform = transforms.Compose([
        transforms.Resize((224, 224)),  # DenseNet expects 224x224
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    dataset = DeepfakeDataset(real_dirs, fake_dirs, transform)
    if len(dataset) == 0: return

    # Split 80% Train, 20% Validation
    val_size = int(0.2 * len(dataset))
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    model = build_densenet_detector().to(device)

    # BCEWithLogitsLoss combines Sigmoid and BCELoss for better numerical stability
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    print("\nStarting Training...")
    best_acc = 0.0

    for epoch in range(epochs):
        start_time = time.time()

        # --- Training Phase ---
        model.train()
        train_loss, train_correct = 0.0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            preds = torch.sigmoid(outputs) >= 0.5
            train_correct += (preds == labels).sum().item()

        train_loss = train_loss / train_size
        train_acc = train_correct / train_size

        # --- Validation Phase ---
        model.eval()
        val_loss, val_correct = 0.0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                preds = torch.sigmoid(outputs) >= 0.5
                val_correct += (preds == labels).sum().item()

        val_loss = val_loss / val_size
        val_acc = val_correct / val_size

        epoch_time = time.time() - start_time
        print(f"Epoch {epoch + 1:02d}/{epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | Time: {epoch_time:.0f}s")

        # Save the best model
        if val_acc > best_acc:
            best_acc = val_acc
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            torch.save(model.state_dict(), save_path)
            print(f"--> Saved new best model (Val Acc: {best_acc:.4f})")

    print(f"\nTraining Complete. Best Validation Accuracy: {best_acc:.4f}")
    print(f"Model saved to: {save_path}")

Execution Trigger

In [ ]:
# 1. Map the folders
real_folders = [
    "../data/raw_mri/real_bright",
    "../data/raw_mri/real_dark"
]

fake_folders = [
    "../data/synthetic_gans/bright",
    "../data/synthetic_gans/dark"
]

# 2. Output Path
detector_save_path = os.path.join("../saved_models", "DenseNet121_Detector.pth")

# 3. Run Training
train_detector(real_folders, fake_folders, detector_save_path, epochs=20, batch_size=32)